# Realism Check and Graded Comfort Metric

This notebook demos the evaluation artifact `eval.py`, which re-analyzes (read-only, no new predictor training) two prior artifacts:

- a **synthetic** room-transition-vs-PreHeat thermal-simulation experiment (`method.py`), which predicts which rooms will be occupied a few minutes ahead so a heater can pre-heat them, and
- the **real CASAS** multi-room occupancy dataset (Aruba/Cairo/Milan/Tulum houses).

It computes four metric families:

1. **Realism/calibration** — how close the synthetic occupant-trajectory generator is to real CASAS households (occupied fraction, dwell time, transition entropy, z-scored Euclidean distance).
2. **Corrected bootstrap CI + power analysis** on the original energy-savings result, showing the small (n=topologies) design is underpowered rather than a confirmed null.
3. **Graded comfort proxy** — a fine-grained re-simulation that measures integrated temperature deficit and anticipation lead time, bypassing the original binary MissTime metric's saturation.
4. **Full AUC table** — all topology x lookahead ROC-AUC cells recomputed from stored ROC curves, with a paired significance test.

**Minimal-demo scale**: the full run uses 3 household topologies x 60 days, 4 lookaheads (15-60min), 3 FPR targets, and 2000 bootstrap resamples. This notebook uses a pre-computed *mini* dataset with **1 topology x 14 days, 2 lookaheads, 2 FPR targets, 200 bootstrap resamples**, and a small slice (3 days x all rooms) of each of the 4 real CASAS houses — small enough to run in well under a minute, while exercising every code path of the original script unchanged.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru, statsmodels -- NOT pre-installed on Colab, always install
_pip('loguru==0.7.3')
_pip('statsmodels==0.14.6')

# numpy, scipy, scikit-learn -- pre-installed on Colab, install locally only (exact Colab versions)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'scikit-learn==1.6.1', 'matplotlib==3.10.0')


In [ ]:
from __future__ import annotations

import ast
import json
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
from loguru import logger
from scipy import stats
from sklearn.metrics import auc as sk_auc
from statsmodels.stats.power import TTestPower

import sys as _sys
logger.remove()
logger.add(_sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")


## Load demo data

`mini_demo_data.json` bundles a small pre-computed slice of both dependency artifacts, so this notebook does not need to re-run the (expensive, multiprocessing) original `method.py` pipeline from scratch:

- `method_out` — the same JSON structure `method.py` writes (`metadata.per_topology_results`, `metadata.fpr_targets`, `metadata.lookaheads_min`, `metadata.aggregate_by_fpr`), but for **1 topology x 14 days** instead of 3 topologies x 60 days, 2 lookaheads instead of 4, 2 FPR targets instead of 3.
- `casas_examples` — a 3-day slice (all rooms) of each of the 4 real CASAS houses, in the same per-example schema as `full_data_out.json`.

It is loaded from GitHub with a local-file fallback so the notebook works both standalone and after being pushed to the repo.

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-ba45a6-transition-sequence-pre-heating-beats/main/round-2/evaluation-1/demo/mini_demo_data.json"
import os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")


In [ ]:
data = load_data()
method_out = data["method_out"]
casas_examples = data["casas_examples"]
print("topologies:", [t["topology"] for t in method_out["metadata"]["per_topology_results"]])
print("lookaheads_min:", method_out["metadata"]["lookaheads_min"], " fpr_targets:", method_out["metadata"]["fpr_targets"])
print("casas houses:", sorted({ex["metadata_house_id"] for ex in casas_examples}))
print("n casas examples:", len(casas_examples))


## Config

All tunable parameters (originally module-level constants in `method.py` / `eval.py`), set to the demo-mini scale recorded in `data["config"]`. The commented-out values are the originals from the full run -- raise `N_BOOT` and swap in the full `mini_demo_data.json` -> `full_method_out.json`/`full_data_out.json` pair to reproduce them.

In [ ]:
# -- method.py / eval.py constants --
SLOTS_PER_DAY = 96  # 15-min resolution
DT_MIN = 15.0
DT_HOURS = DT_MIN / 60.0
TARGET_TEMP = method_out["metadata"]["target_temp_c"]  # 20.0
SETBACK_MISS_TOL = method_out["metadata"]["miss_tolerance_c"]  # 1.0
K_COUPLING = 30.0  # W/C, adjacent-room thermal coupling
RNG_SEED = data["config"]["rng_seed"]
AWAY = "AWAY"

# -- demo-mini scale (see data["config"]) --
N_DAYS = data["config"]["n_days"]                  # 14   (full run: 60)
LOOKAHEADS = method_out["metadata"]["lookaheads_min"]  # [15, 30]  (full run: [15, 30, 45, 60])
FPR_TARGETS = method_out["metadata"]["fpr_targets"]    # [0.05, 0.2]  (full run: [0.05, 0.1, 0.2])
N_BOOT = data["config"]["n_bootstrap"]             # 200  (full run: 2000)


## `method.py`'s own code (imported directly by `eval.py`, not reimplemented)

The evaluation script re-derives synthetic trajectories and re-runs the fine-grained forward simulation by importing `method.py`'s own `Topology`, `generate_topology_data`, `PreHeatPredictor`, `TransitionPredictor`, `init_thermal_models`, `thermal_step`, `fit_heat_rate`, etc. directly (`import method as M`), with the exact seeds/config recorded in `full_method_out.json` -- nothing about the simulator itself is reimplemented by the evaluation. The cell below is that same code, copied in verbatim (only the module-level constants above are pulled from config instead of being hardcoded) so this notebook is self-contained.

In [ ]:
from dataclasses import dataclass


@dataclass
class Topology:
    name: str
    rooms: list[str]  # excludes "AWAY"
    adjacency: dict[str, list[str]]  # room -> neighbor rooms (thermal coupling)
    regularity: float  # 0=irregular (near-uniform transitions), 1=highly regular
    n_days: int = 60


def make_topologies() -> list[Topology]:
    """3 synthetic household topologies spanning room-count / adjacency / regularity."""
    return [
        Topology(
            name="studio_linear_regular",
            rooms=["bedroom", "living", "kitchen"],
            adjacency={"bedroom": ["living"], "living": ["bedroom", "kitchen"], "kitchen": ["living"]},
            regularity=0.85,
            n_days=60,
        ),
        Topology(
            name="apartment_star_mixed",
            rooms=["bedroom", "hallway", "living", "kitchen"],
            adjacency={
                "bedroom": ["hallway"],
                "hallway": ["bedroom", "living", "kitchen"],
                "living": ["hallway"],
                "kitchen": ["hallway"],
            },
            regularity=0.55,
            n_days=60,
        ),
        Topology(
            name="house_5room_irregular",
            rooms=["bedroom1", "bedroom2", "hallway", "living", "kitchen"],
            adjacency={
                "bedroom1": ["hallway"],
                "bedroom2": ["hallway"],
                "hallway": ["bedroom1", "bedroom2", "living", "kitchen"],
                "living": ["hallway", "kitchen"],
                "kitchen": ["hallway", "living"],
            },
            regularity=0.25,
            n_days=60,
        ),
    ]


def _room_index(topo: Topology) -> dict[str, int]:
    labels = [AWAY] + topo.rooms
    return {r: i for i, r in enumerate(labels)}


def _preferred_daily_schedule(rng: np.random.Generator, topo: Topology, daytype: str) -> list[tuple[str, float]]:
    """A loose 'preferred' sequence of (room, mean_dwell_minutes) anchoring the regular
    component of the semi-Markov walk. Weekday vs weekend differ (PreHeat's own split)."""
    rooms = topo.rooms
    if daytype == "weekday":
        anchors = [
            (AWAY, 0, 420),  # midnight-7am: asleep -> counted as "bedroom" not away; handled below
            ("bedroom", 0, 420),
            ("kitchen", 420, 60),
            (AWAY, 480, 540),  # away at work
            ("kitchen", 1020, 60),
            ("living", 1080, 300),
            ("bedroom", 1380, 60),
        ]
    else:
        anchors = [
            ("bedroom", 0, 480),
            ("kitchen", 480, 45),
            ("living", 525, 360),
            ("kitchen", 885, 60),
            ("living", 945, 300),
            (AWAY, 1245, 120),
            ("bedroom", 1365, 75),
        ]
    out = []
    for room, start_min, dur_min in anchors:
        if room != AWAY and room not in rooms:
            room = rooms[hash(room) % len(rooms)]
        out.append((room, start_min, dur_min))
    return out


def generate_topology_data(topo: Topology, seed: int) -> dict:
    """Semi-Markov random walk occupant simulator -> per-room binary occupancy matrices."""
    rng = np.random.default_rng(seed)
    idx = _room_index(topo)

    occ = {r: np.zeros((topo.n_days, SLOTS_PER_DAY), dtype=np.int8) for r in topo.rooms}
    trajectories = []  # list of (daytype, [room_label per slot])
    daytypes = []

    for day in range(topo.n_days):
        dow = day % 7
        daytype = "weekend" if dow >= 5 else "weekday"
        schedule = _preferred_daily_schedule(rng, topo, daytype)
        # regular component: sample dwell durations around the anchor durations with
        # log-normal jitter; irregular component: with prob (1-regularity) at each
        # anchor, substitute a uniformly random room/duration instead.
        slots = np.empty(SLOTS_PER_DAY, dtype=object)
        cur_min = 0
        for anchor_i, (room, start_min, dur_min) in enumerate(schedule):
            if cur_min >= 1440:
                break
            use_regular = rng.random() < topo.regularity
            if not use_regular:
                room = rng.choice([AWAY] + topo.rooms)
                dur_min = max(15.0, rng.lognormal(mean=np.log(90), sigma=0.9))
            else:
                jitter = rng.lognormal(mean=0.0, sigma=0.25)
                dur_min = max(15.0, dur_min * jitter if dur_min > 0 else 30.0)
            end_min = min(1440, cur_min + dur_min)
            s0, s1 = int(cur_min // DT_MIN), int(end_min // DT_MIN)
            s1 = max(s1, s0 + 1)
            for s in range(s0, min(s1, SLOTS_PER_DAY)):
                slots[s] = room
            cur_min = end_min
        # fill any trailing gap by repeating the last room
        last = topo.rooms[0]
        for s in range(SLOTS_PER_DAY):
            if slots[s] is None:
                slots[s] = last
            else:
                last = slots[s]

        # sensor noise: 5-10% independent flip rate per (room, slot)
        noise_rate = rng.uniform(0.05, 0.10)
        for room in topo.rooms:
            true_bits = (slots == room).astype(np.int8)
            flips = rng.random(SLOTS_PER_DAY) < noise_rate
            noisy = np.where(flips, 1 - true_bits, true_bits)
            occ[room][day] = noisy

        trajectories.append([str(x) for x in slots])
        daytypes.append(daytype)

    return {"occ": occ, "trajectories": trajectories, "daytypes": daytypes, "idx": idx}


def load_or_synthesize_weather(topo: Topology, seed: int) -> np.ndarray:
    """Outdoor temperature trace (n_days x 96), diurnal sinusoid + AR(1) day-to-day drift."""
    rng = np.random.default_rng(seed + 777)
    base = 6.0  # UK winter-ish mean outdoor temp C
    hours = np.arange(SLOTS_PER_DAY) * DT_MIN / 60.0
    diurnal = -4.0 * np.cos(2 * np.pi * (hours - 4) / 24.0)  # coldest ~4am, warmest ~4pm
    trace = np.zeros((topo.n_days, SLOTS_PER_DAY))
    level = base
    for d in range(topo.n_days):
        level = 0.9 * level + 0.1 * base + rng.normal(0, 1.2)
        trace[d] = level + diurnal + rng.normal(0, 0.3, SLOTS_PER_DAY)
    return trace


In [ ]:
class PreHeatPredictor:
    """Reproduction of PreHeat (Scott et al. 2011): for each room, find the K=5
    historical days of the same day-type (weekday/weekend) whose occupancy so far
    today most closely matches (min Hamming distance), then average their future
    occupancy as the probability forecast."""

    K = 5

    def __init__(self):
        self.history: dict[str, dict[str, np.ndarray]] = {}  # room -> daytype -> (n_days,96)

    def fit(self, occ: dict[str, np.ndarray], daytypes: list[str], train_idx: list[int]) -> None:
        dt_arr = np.array(daytypes)
        for room, mat in occ.items():
            self.history[room] = {
                "weekday": mat[train_idx][dt_arr[train_idx] == "weekday"],
                "weekend": mat[train_idx][dt_arr[train_idx] == "weekend"],
            }

    def predict_curve(self, room: str, partial_today: np.ndarray, slot_idx: int, daytype: str, max_lookahead_slots: int) -> np.ndarray:
        """Return probability-occupied for slots [slot_idx+1 .. slot_idx+max_lookahead_slots]."""
        cands = self.history[room][daytype]
        if len(cands) == 0 or slot_idx == 0:
            return np.full(max_lookahead_slots, float(np.mean(cands[:, slot_idx + 1:slot_idx + 1 + max_lookahead_slots])) if len(cands) else 0.5)
        known = partial_today[:slot_idx]
        cand_known = cands[:, :slot_idx]
        dists = np.sum(known[None, :] != cand_known, axis=1)
        k = min(self.K, len(cands))
        top = np.argsort(dists, kind="stable")[:k]
        future_slices = cands[top, slot_idx:slot_idx + max_lookahead_slots]
        n_have = future_slices.shape[1]
        curve = np.mean(future_slices, axis=0)
        if n_have < max_lookahead_slots:
            curve = np.pad(curve, (0, max_lookahead_slots - n_have), constant_values=curve[-1] if n_have else 0.5)
        return curve


class TransitionPredictor:
    """Order-1 slot-to-slot Markov chain over {AWAY, room1..N} (weekday/weekend split),
    with an order-2 (last-two-slots) context model with additive-smoothing backoff
    for the immediate next-slot prediction, then closed-form matrix-power propagation
    for longer horizons (fallback_plan item 3: avoids Monte-Carlo rollout noise)."""

    def __init__(self, labels: list[str]):
        self.labels = labels
        self.lidx = {l: i for i, l in enumerate(labels)}
        self.n = len(labels)
        self.T1: dict[str, np.ndarray] = {}
        self.T2counts: dict[str, dict[tuple[str, str], np.ndarray]] = {}

    def fit(self, trajectories: list[list[str]], daytypes: list[str], train_idx: list[int]) -> None:
        for daytype in ["weekday", "weekend"]:
            counts1 = np.ones((self.n, self.n)) * 0.5  # additive smoothing
            counts2: dict[tuple[str, str], np.ndarray] = {}
            for i in train_idx:
                if daytypes[i] != daytype:
                    continue
                traj = trajectories[i]
                for t in range(len(traj) - 1):
                    a, b = self.lidx[traj[t]], self.lidx[traj[t + 1]]
                    counts1[a, b] += 1
                    if t >= 1:
                        ctx = (traj[t - 1], traj[t])
                        if ctx not in counts2:
                            counts2[ctx] = np.zeros(self.n)
                        counts2[ctx][b] += 1
            T1 = counts1 / counts1.sum(axis=1, keepdims=True)
            self.T1[daytype] = T1
            self.T2counts[daytype] = counts2

    def _next_slot_dist(self, prev2: str | None, prev1: str, daytype: str) -> np.ndarray:
        T1 = self.T1[daytype]
        base = T1[self.lidx[prev1]]
        if prev2 is None:
            return base
        ctx = (prev2, prev1)
        c2 = self.T2counts[daytype].get(ctx)
        if c2 is None or c2.sum() < 3:  # backoff: too little context evidence
            return base
        alpha = 3.0  # additive smoothing strength for backoff blending
        blended = (c2 + alpha * base) / (c2.sum() + alpha)
        return blended

    def predict_curve(self, room: str, traj_so_far: list[str], daytype: str, max_lookahead_slots: int) -> np.ndarray:
        prev1 = traj_so_far[-1]
        prev2 = traj_so_far[-2] if len(traj_so_far) >= 2 else None
        dist_next = self._next_slot_dist(prev2, prev1, daytype)
        T1 = self.T1[daytype]
        room_i = self.lidx[room]
        curve = np.empty(max_lookahead_slots)
        dist = dist_next
        curve[0] = dist[room_i]
        for k in range(1, max_lookahead_slots):
            dist = dist @ T1
            curve[k] = dist[room_i]
        return curve


def compute_roc_points(probs: np.ndarray, labels: np.ndarray, n_thresholds: int = 101) -> dict:
    thresholds = np.linspace(0.0, 1.0, n_thresholds)
    P = max(int(labels.sum()), 1)
    N = max(int((1 - labels).sum()), 1)
    tprs, fprs = [], []
    for th in thresholds:
        pred = probs >= th
        tp = np.sum(pred & (labels == 1))
        fp = np.sum(pred & (labels == 0))
        tprs.append(tp / P)
        fprs.append(fp / N)
    tprs, fprs = np.array(tprs), np.array(fprs)
    order = np.argsort(fprs)
    auc = float(np.trapezoid(tprs[order], fprs[order]))
    return {"thresholds": thresholds.tolist(), "tpr": tprs.tolist(), "fpr": fprs.tolist(), "auc": auc}


def threshold_at_fpr(roc: dict, target_fpr: float) -> float:
    fprs = np.array(roc["fpr"])
    thresholds = np.array(roc["thresholds"])
    order = np.argsort(thresholds)  # descending threshold -> ascending fpr generally
    fprs_o, th_o = fprs[order], thresholds[order]
    idx = np.searchsorted(fprs_o, target_fpr, side="left")
    idx = min(idx, len(th_o) - 1)
    return float(th_o[idx])


In [ ]:
@dataclass
class RoomThermal:
    C: float  # thermal mass, kWh/C
    U: float  # heat loss coeff, W/C
    Q_max: float  # heater max output, W
    T: float = 18.0


def init_thermal_models(topo: Topology, rng: np.random.Generator) -> dict[str, RoomThermal]:
    models = {}
    for r in topo.rooms:
        models[r] = RoomThermal(
            C=rng.uniform(2.0, 6.0),
            U=rng.uniform(120.0, 260.0),
            Q_max=rng.uniform(1200.0, 2000.0),
            T=rng.uniform(15.0, 18.0),
        )
    return models


def thermal_step(models: dict[str, RoomThermal], adjacency: dict[str, list[str]], heater_on: dict[str, bool], T_out: float) -> None:
    temps_prev = {r: m.T for r, m in models.items()}
    for r, m in models.items():
        Q = m.Q_max if heater_on[r] else 0.0
        loss = m.U * (temps_prev[r] - T_out)
        coupling = sum(K_COUPLING * (temps_prev[r] - temps_prev[n]) for n in adjacency.get(r, []))
        dT = (Q - loss - coupling) * DT_HOURS / (m.C * 1000.0)  # C is kWh/C -> Wh/C = C*1000
        m.T = temps_prev[r] + dT


def fit_heat_rate(models_template: dict[str, RoomThermal], adjacency: dict[str, list[str]], T_out: float = 5.0) -> dict[str, float]:
    """Empirically fit avg C/min heat rate per room from a short constant-heating calibration run."""
    import copy

    calib = copy.deepcopy(models_template)
    n_steps = 8  # 2 hours
    T0 = {r: m.T for r, m in calib.items()}
    for _ in range(n_steps):
        thermal_step(calib, adjacency, {r: True for r in calib}, T_out)
    return {r: max((calib[r].T - T0[r]) / (n_steps * DT_MIN), 1e-4) for r in calib}


def run_simulation(
    predictor_kind: str,
    threshold: float,
    topo: Topology,
    data_: dict,
    weather: np.ndarray,
    test_idx: list[int],
    lookahead_slots: int,
    baseline: PreHeatPredictor | None,
    transition: TransitionPredictor | None,
    rng: np.random.Generator,
) -> dict:
    models = init_thermal_models(topo, rng)
    heat_rate = fit_heat_rate(models, topo.adjacency)
    gas_used_wh = 0.0
    miss_time_min = 0.0
    occ = data_["occ"]
    trajectories = data_["trajectories"]
    daytypes = data_["daytypes"]

    for day in test_idx:
        daytype = daytypes[day]
        traj = trajectories[day]
        for slot in range(SLOTS_PER_DAY):
            T_out = weather[day, slot]
            heater_on = {}
            for room in topo.rooms:
                true_occ_now = bool(occ[room][day, slot])
                if predictor_kind == "scheduled":
                    predicted_occupied = 6 * 4 <= slot <= 22 * 4  # 06:00-22:00 fixed schedule
                elif predictor_kind == "reactive":
                    predicted_occupied = true_occ_now
                elif predictor_kind == "preheat":
                    if slot == 0:
                        prob = 0.5
                    else:
                        partial = occ[room][day]
                        curve = baseline.predict_curve(room, partial, slot, daytype, lookahead_slots)
                        prob = curve[-1]
                    predicted_occupied = prob >= threshold
                elif predictor_kind == "transition":
                    if slot == 0:
                        prob = 0.5
                    else:
                        curve = transition.predict_curve(room, traj[:slot], daytype, lookahead_slots)
                        prob = curve[-1]
                    predicted_occupied = prob >= threshold
                else:
                    raise ValueError(predictor_kind)

                m = models[room]
                heat_ahead = predicted_occupied and (m.T + heat_rate[room] * lookahead_slots * DT_MIN < TARGET_TEMP)
                reactive_fallback = true_occ_now and m.T < TARGET_TEMP
                heater_on[room] = bool(heat_ahead or reactive_fallback)

            thermal_step(models, topo.adjacency, heater_on, T_out)
            for room in topo.rooms:
                if heater_on[room]:
                    gas_used_wh += models[room].Q_max * DT_HOURS
                if bool(occ[room][day, slot]) and models[room].T < TARGET_TEMP - SETBACK_MISS_TOL:
                    miss_time_min += DT_MIN

    return {"gas_proxy_wh": gas_used_wh, "miss_time_min": miss_time_min}


def bootstrap_savings_ci(topo_results: list[dict], target_fpr: float, n_boot: int, seed: int) -> dict:
    """Per PLAN step 6: bootstrap over topologies (each topology contributes one
    savings-% point at this FPR; day-level resampling is embedded in each topology's
    already-computed simulation, so here we bootstrap-resample the topology-level
    savings estimates themselves to get a cross-topology CI)."""
    rng = np.random.default_rng(seed)
    savings = []
    miss_deltas = []
    for tr in topo_results:
        entry = next(e for e in tr["fpr_matched_results"] if e["target_fpr"] == target_fpr)
        p, t = entry["preheat"], entry["transition"]
        sav = 100.0 * (p["gas_proxy_wh"] - t["gas_proxy_wh"]) / max(p["gas_proxy_wh"], 1e-9)
        savings.append(sav)
        miss_deltas.append(t["miss_time_min"] - p["miss_time_min"])
    savings = np.array(savings)
    miss_deltas = np.array(miss_deltas)
    n = len(savings)
    boot_means = np.array([savings[rng.integers(0, n, n)].mean() for _ in range(n_boot)])
    ci_lo, ci_hi = float(np.percentile(boot_means, 2.5)), float(np.percentile(boot_means, 97.5))
    return {
        "target_fpr": target_fpr,
        "per_topology_savings_pct": savings.tolist(),
        "mean_savings_pct": float(savings.mean()),
        "bootstrap_ci95": [ci_lo, ci_hi],
        "ci_excludes_zero": bool(ci_lo > 0 or ci_hi < 0),
        "mean_miss_time_delta_min": float(miss_deltas.mean()),
        "miss_time_noninferior": bool(np.all(miss_deltas <= 5.0)),  # tolerance: <=5min/day-agg extra miss
    }


## (1) Realism / calibration: synthetic vs real CASAS

`dwell_stats`, `transition_entropy_bits`, and `casas_room_sequences_for_day` are `eval.py`'s own helpers, unchanged. `compute_realism_table` is `eval.py`'s function with two changes: it reads `method_out`/`casas_examples` from the loaded `data` dict instead of `full_method_out.json`/`full_data_out.json`, and it re-derives synthetic trajectories with the demo-mini `topo.n_days` (14, not 60) already baked into `topo_by_name` from `method_out`.

In [ ]:
def dwell_stats(vec: np.ndarray, bin_min: float) -> tuple[float, float]:
    """occupied_fraction, mean_dwell_minutes for a binary occupancy vector."""
    occ_frac = float(np.mean(vec))
    runs = []
    cur = 0
    for b in vec:
        if b:
            cur += 1
        else:
            if cur > 0:
                runs.append(cur)
            cur = 0
    if cur > 0:
        runs.append(cur)
    mean_dwell = float(np.mean(runs) * bin_min) if runs else 0.0
    return occ_frac, mean_dwell


def transition_entropy_bits(room_sequences: list[list[str]]) -> float:
    """H(next_room | current_room) in bits, visit-count-weighted, from a list of
    per-day room-label sequences (one label per bin, including an AWAY-equivalent)."""
    counts: dict[str, dict[str, int]] = defaultdict(lambda: defaultdict(int))
    for seq in room_sequences:
        for a, b in zip(seq[:-1], seq[1:]):
            if a == b:
                continue  # only count actual room-to-room transitions, not self-dwell
            counts[a][b] += 1
    total_visits = 0
    weighted_h = 0.0
    for src, dsts in counts.items():
        n = sum(dsts.values())
        if n == 0:
            continue
        p = np.array(list(dsts.values()), dtype=float) / n
        h = float(-np.sum(p * np.log2(p)))
        weighted_h += h * n
        total_visits += n
    return weighted_h / total_visits if total_visits > 0 else 0.0


def casas_room_sequences_for_day(transitions_capped: list) -> list[str]:
    """Reconstruct an ordered room-label sequence for one house-day from the
    (from_room, to_room, time, dwell_min) transition tuples already attached to
    each CASAS example (capped at 40)."""
    if not transitions_capped:
        return []
    seq = [transitions_capped[0][0]]
    for tr in transitions_capped:
        seq.append(tr[1])
    return seq


def compute_realism_table() -> dict:
    logger.info("[1] Realism check: synthetic topologies vs real CASAS houses")
    topo_results = method_out["metadata"]["per_topology_results"]

    # re-derive synthetic trajectories via method.py's own generator, same seeds as main()
    topologies = make_topologies()
    topo_by_name = {t.name: t for t in topologies}
    seed_by_name = {t.name: RNG_SEED + i * 1000 for i, t in enumerate(topologies)}

    rows = []
    synth_vecs = {}
    for tr in topo_results:
        name = tr["topology"]
        topo = topo_by_name[name]
        topo.n_days = N_DAYS  # demo-mini scale (14, not 60)
        seed = seed_by_name[name]
        gen = generate_topology_data(topo, seed)
        occ, trajectories = gen["occ"], gen["trajectories"]

        occ_fracs, dwells = [], []
        for room in topo.rooms:
            mat = occ[room]  # (n_days, 96)
            for day in range(mat.shape[0]):
                f, d = dwell_stats(mat[day], DT_MIN)
                occ_fracs.append(f)
                dwells.append(d)
        ent = transition_entropy_bits(trajectories)
        row = {
            "row": f"synthetic_{name}",
            "occupied_fraction": float(np.mean(occ_fracs)),
            "mean_dwell_min": float(np.mean(dwells)),
            "transition_entropy_bits": ent,
            "n_rooms": len(topo.rooms),
            "n_days": topo.n_days,
        }
        rows.append(row)
        synth_vecs[name] = np.array(
            [row["occupied_fraction"], row["mean_dwell_min"], row["transition_entropy_bits"]]
        )
        logger.info(f"  synthetic/{name}: {row}")
        del gen, occ, trajectories

    # real CASAS houses
    casas_by_house: dict[str, list] = defaultdict(list)
    for ex in casas_examples:
        casas_by_house[ex["metadata_house_id"]].append(ex)

    casas_vecs = {}
    for house, examples in casas_by_house.items():
        occ_fracs, dwells = [], []
        seqs_by_day: dict[str, list] = {}
        for ex in examples:
            vec = np.array(ast.literal_eval(ex["input"]), dtype=np.int8)
            f, d = dwell_stats(vec, ex["metadata_bin_minutes"])
            occ_fracs.append(f)
            dwells.append(d)
            day_id = ex["metadata_day_id"]
            if day_id not in seqs_by_day:
                seqs_by_day[day_id] = casas_room_sequences_for_day(ex["metadata_room_transitions"])
        seqs = [s for s in seqs_by_day.values() if len(s) >= 2]
        ent = transition_entropy_bits(seqs) if seqs else 0.0
        n_days = len({ex["metadata_day_id"] for ex in examples})
        row = {
            "row": f"CASAS_{house}",
            "occupied_fraction": float(np.mean(occ_fracs)),
            "mean_dwell_min": float(np.mean(dwells)),
            "transition_entropy_bits": ent,
            "n_rooms": len({ex["metadata_room_id"] for ex in examples}),
            "n_days": n_days,
        }
        rows.append(row)
        casas_vecs[house] = np.array([row["occupied_fraction"], row["mean_dwell_min"], row["transition_entropy_bits"]])
        logger.info(f"  CASAS/{house}: {row}")

    # z-score the 3 quantities across all rows, then Euclidean distance matrix
    all_mat = np.array([[r["occupied_fraction"], r["mean_dwell_min"], r["transition_entropy_bits"]] for r in rows])
    mu, sd = all_mat.mean(axis=0), all_mat.std(axis=0)
    sd = np.where(sd < 1e-12, 1.0, sd)
    z = (all_mat - mu) / sd
    z_by_row = {r["row"]: z[i] for i, r in enumerate(rows)}

    synth_names = list(synth_vecs.keys())
    house_names = list(casas_vecs.keys())
    dist_matrix = []
    for sname in synth_names:
        zs = z_by_row[f"synthetic_{sname}"]
        drow = {}
        for hname in house_names:
            zh = z_by_row[f"CASAS_{hname}"]
            drow[hname] = float(np.linalg.norm(zs - zh))
        dist_matrix.append({"synthetic_topology": sname, "distances_to_house": drow})

    closest = min(
        ((s["synthetic_topology"], h, d) for s in dist_matrix for h, d in s["distances_to_house"].items()),
        key=lambda x: x[2],
    )
    farthest = max(
        ((s["synthetic_topology"], h, d) for s in dist_matrix for h, d in s["distances_to_house"].items()),
        key=lambda x: x[2],
    )

    return {
        "table": rows,
        "synthetic_to_real_distance_matrix": dist_matrix,
        "closest_synthetic_to_real_pair": {"synthetic": closest[0], "real": closest[1], "z_distance": closest[2]},
        "farthest_synthetic_to_real_pair": {"synthetic": farthest[0], "real": farthest[1], "z_distance": farthest[2]},
    }


## (2) Corrected bootstrap CI + power analysis

Re-runs `method.py`'s own `bootstrap_savings_ci` at `N_BOOT` resamples and compares to the CI already stored in `method_out`, then runs a post-hoc power analysis (one-sample t-test MDE formula plus `statsmodels.TTestPower.solve_power`) asking how many independent topology conditions would be needed to reliably detect a 5-percentage-point savings effect. With only 1 demo topology, `n_topologies=1` here (vs. 3 in the full run), so the CI is necessarily very wide -- this cell exists to show the *mechanism* (recompute + power formula), not to reproduce the full-run numbers.

In [ ]:
def compute_ci_power_table() -> dict:
    logger.info("[2] Corrected bootstrap CI + post-hoc power analysis on energy savings")
    topo_results = method_out["metadata"]["per_topology_results"]
    fpr_targets = method_out["metadata"]["fpr_targets"]
    original_agg = {a["target_fpr"]: a for a in method_out["metadata"]["aggregate_by_fpr"]}

    rows = []
    for target_fpr in fpr_targets:
        recomputed = bootstrap_savings_ci(topo_results, target_fpr, n_boot=N_BOOT, seed=RNG_SEED)
        orig = original_agg[target_fpr]
        savings = np.array(recomputed["per_topology_savings_pct"])
        n = len(savings)
        sd = float(np.std(savings, ddof=1))
        alpha, power_target = 0.05, 0.80

        t_alpha = stats.t.ppf(1 - alpha / 2, df=n - 1)
        t_beta = stats.t.ppf(power_target, df=n - 1)
        mde_n3 = float((t_alpha + t_beta) * sd / np.sqrt(n))

        power_solver = TTestPower()
        observed_mean = float(np.mean(savings))
        eff_5pp = 5.0 / sd if sd > 1e-9 else float("inf")
        n_req_5pp = float(power_solver.solve_power(effect_size=eff_5pp, alpha=alpha, power=power_target, alternative="two-sided")) if sd > 1e-9 else float("nan")

        eff_observed = observed_mean / sd if sd > 1e-9 else float("inf")
        n_req_observed = (
            float(power_solver.solve_power(effect_size=eff_observed, alpha=alpha, power=power_target, alternative="two-sided"))
            if abs(eff_observed) > 1e-9
            else float("nan")
        )

        rows.append(
            {
                "target_fpr": target_fpr,
                "recomputed_bootstrap_ci95": recomputed["bootstrap_ci95"],
                "original_bootstrap_ci95": orig["bootstrap_ci95"],
                "ci_reproduces_within_mc_noise": bool(
                    abs(recomputed["bootstrap_ci95"][0] - orig["bootstrap_ci95"][0]) < 1.0
                    and abs(recomputed["bootstrap_ci95"][1] - orig["bootstrap_ci95"][1]) < 1.0
                ),
                "mean_savings_pct": observed_mean,
                "observed_sd_pct": sd,
                "n_topologies": n,
                "mde_n3_at_80pct_power_pp": mde_n3,
                "n_required_for_5pp_effect_at_80pct_power": n_req_5pp,
                "n_required_for_observed_magnitude_at_80pct_power": n_req_observed,
            }
        )
        logger.info(f"  FPR={target_fpr}: recomputed_ci={recomputed['bootstrap_ci95']} sd={sd:.2f} MDE(n={n})={mde_n3:.2f}pp n_req(5pp)={n_req_5pp:.1f}")

    # companion larger-n experiment: not a dependency of this artifact -- fallback per plan
    companion_available = False
    companion_note = "companion experiment not available at eval time -- reporting power analysis at demo-mini n only"
    logger.info(f"  companion expanded-topology experiment: {companion_note}")

    return {"table": rows, "companion_larger_n_available": companion_available, "companion_note": companion_note}


## (3) Graded comfort proxy: fine-grained re-simulation

Re-runs the forward thermal simulation (reusing `init_thermal_models`/`thermal_step`/`fit_heat_rate`/`predictor.predict_curve` verbatim from `method.py`, only the surrounding bookkeeping is new) to compute two metrics invisible to the original binary MissTime metric: (a) integrated temperature deficit during windows the predictor anticipated occupancy but it hadn't happened yet, and (b) realized anticipation lead time at each true occupancy onset.

In [ ]:
def fine_grained_simulation(
    predictor_kind: str,
    threshold: float,
    topo: Topology,
    data_: dict,
    weather: np.ndarray,
    test_idx: list[int],
    lookahead_slots: int,
    baseline,
    transition,
    rng: np.random.Generator,
) -> dict:
    """Re-run of method.py's run_simulation forward pass, augmented to also record
    (a) integrated temperature deficit during anticipated-but-not-yet-occupied windows,
    (b) realized anticipation lead time per true occupancy-onset event.
    Reuses init_thermal_models / thermal_step / fit_heat_rate / predictor.predict_curve
    from method.py verbatim -- only the bookkeeping around them is new."""
    models = init_thermal_models(topo, rng)
    heat_rate = fit_heat_rate(models, topo.adjacency)
    occ = data_["occ"]
    trajectories = data_["trajectories"]
    daytypes = data_["daytypes"]

    integrated_deficit_degc_min = 0.0
    lead_times_min: list[float] = []
    n_onset_events = 0
    n_positive_lead = 0

    for day in test_idx:
        daytype = daytypes[day]
        traj = trajectories[day]
        # per-room: track when the predictor's forecast first crosses `threshold`
        # before the current onset run, to compute lead time at onset.
        crossed_at_slot: dict[str, int | None] = {room: None for room in topo.rooms}
        prev_true_occ: dict[str, bool] = {room: False for room in topo.rooms}

        for slot in range(SLOTS_PER_DAY):
            T_out = weather[day, slot]
            heater_on = {}
            predicted_occupied_map = {}
            for room in topo.rooms:
                true_occ_now = bool(occ[room][day, slot])
                if predictor_kind == "preheat":
                    if slot == 0:
                        prob = 0.5
                    else:
                        partial = occ[room][day]
                        curve = baseline.predict_curve(room, partial, slot, daytype, lookahead_slots)
                        prob = curve[-1]
                    predicted_occupied = prob >= threshold
                elif predictor_kind == "transition":
                    if slot == 0:
                        prob = 0.5
                    else:
                        curve = transition.predict_curve(room, traj[:slot], daytype, lookahead_slots)
                        prob = curve[-1]
                    predicted_occupied = prob >= threshold
                else:
                    raise ValueError(predictor_kind)
                predicted_occupied_map[room] = predicted_occupied

                m = models[room]
                heat_ahead = predicted_occupied and (m.T + heat_rate[room] * lookahead_slots * DT_MIN < TARGET_TEMP)
                reactive_fallback = true_occ_now and m.T < TARGET_TEMP
                heater_on[room] = bool(heat_ahead or reactive_fallback)

                # (a) integrated deficit during anticipated-but-not-yet-occupied windows
                if predicted_occupied and not true_occ_now:
                    deficit = max(0.0, TARGET_TEMP - m.T)
                    integrated_deficit_degc_min += deficit * DT_MIN

                # (b) track forecast-crossing slot for lead-time computation
                if predicted_occupied and crossed_at_slot[room] is None and not true_occ_now:
                    crossed_at_slot[room] = slot

                # onset event: transition False -> True in true occupancy
                if true_occ_now and not prev_true_occ[room]:
                    n_onset_events += 1
                    if crossed_at_slot[room] is not None:
                        lead_slots = slot - crossed_at_slot[room]
                        lead_min = lead_slots * DT_MIN
                    else:
                        lead_min = 0.0  # never crossed before onset -> caught only by reactive fallback
                    lead_times_min.append(lead_min)
                    if lead_min > 0:
                        n_positive_lead += 1
                    crossed_at_slot[room] = None  # reset for next occupancy run

                if not true_occ_now:
                    pass  # keep crossed_at_slot until next onset (already reset above at onset)
                prev_true_occ[room] = true_occ_now

            thermal_step(models, topo.adjacency, heater_on, T_out)

    lead_arr = np.array(lead_times_min) if lead_times_min else np.array([0.0])
    return {
        "integrated_deficit_degc_min": integrated_deficit_degc_min,
        "mean_lead_time_min": float(np.mean(lead_arr)),
        "median_lead_time_min": float(np.median(lead_arr)),
        "anticipation_rate": float(n_positive_lead / n_onset_events) if n_onset_events > 0 else 0.0,
        "n_onset_events": n_onset_events,
    }


In [ ]:
def compute_comfort_proxy_table() -> dict:
    logger.info("[3] Graded comfort proxy: integrated deficit + realized lead time (fine-grained re-simulation)")
    topo_results = method_out["metadata"]["per_topology_results"]
    fpr_targets = method_out["metadata"]["fpr_targets"]

    topologies = make_topologies()
    topo_by_name = {t.name: t for t in topologies}
    seed_by_name = {t.name: RNG_SEED + i * 1000 for i, t in enumerate(topologies)}

    rows = []
    by_fpr_pairs: dict[float, list[tuple[dict, dict]]] = defaultdict(list)

    for tr in topo_results:
        name = tr["topology"]
        topo = topo_by_name[name]
        topo.n_days = N_DAYS  # demo-mini scale
        seed = seed_by_name[name]
        gen = generate_topology_data(topo, seed)
        weather = load_or_synthesize_weather(topo, seed)
        occ, trajectories, daytypes = gen["occ"], gen["trajectories"], gen["daytypes"]

        n_days = topo.n_days
        perm = np.random.default_rng(seed + 1).permutation(n_days)
        split = int(n_days * 0.6)
        train_idx = sorted(perm[:split].tolist())
        test_idx = sorted(perm[split:].tolist())

        baseline = PreHeatPredictor()
        baseline.fit(occ, daytypes, train_idx)
        labels_all = [AWAY] + topo.rooms
        transition = TransitionPredictor(labels_all)
        transition.fit(trajectories, daytypes, train_idx)

        sim_lookahead_min = tr["sim_lookahead_min"]
        sim_lookahead_slots = max(1, int(round(sim_lookahead_min / DT_MIN)))
        roc_b = tr["roc_by_lookahead"][str(sim_lookahead_min)]["baseline"]
        roc_t = tr["roc_by_lookahead"][str(sim_lookahead_min)]["transition"]

        sim_rng = np.random.default_rng(seed + 2)
        for target_fpr in fpr_targets:
            th_b = threshold_at_fpr(roc_b, target_fpr)
            th_t = threshold_at_fpr(roc_t, target_fpr)

            res_preheat = fine_grained_simulation(
                "preheat", th_b, topo, gen, weather, test_idx, sim_lookahead_slots, baseline, None, sim_rng
            )
            res_transition = fine_grained_simulation(
                "transition", th_t, topo, gen, weather, test_idx, sim_lookahead_slots, None, transition, sim_rng
            )

            for predictor_name, res in [("PreHeat", res_preheat), ("transition-predictor", res_transition)]:
                rows.append(
                    {
                        "predictor": predictor_name,
                        "topology": name,
                        "target_fpr": target_fpr,
                        "integrated_deficit_degc_min": res["integrated_deficit_degc_min"],
                        "mean_lead_time_min": res["mean_lead_time_min"],
                        "median_lead_time_min": res["median_lead_time_min"],
                        "anticipation_rate": res["anticipation_rate"],
                    }
                )
            by_fpr_pairs[target_fpr].append((res_preheat, res_transition))
            logger.info(
                f"  [{name}] FPR={target_fpr}: PreHeat deficit={res_preheat['integrated_deficit_degc_min']:.1f} "
                f"anticip_rate={res_preheat['anticipation_rate']:.2f} | transition deficit={res_transition['integrated_deficit_degc_min']:.1f} "
                f"anticip_rate={res_transition['anticipation_rate']:.2f}"
            )

        del gen, occ, trajectories, weather
        import gc

        gc.collect()

    # paired difference (transition - PreHeat), per topology-FPR cell + pooled bootstrap CI
    paired_diffs = []
    for tr in topo_results:
        name = tr["topology"]
        for target_fpr in fpr_targets:
            p_row = next(r for r in rows if r["predictor"] == "PreHeat" and r["topology"] == name and r["target_fpr"] == target_fpr)
            t_row = next(r for r in rows if r["predictor"] == "transition-predictor" and r["topology"] == name and r["target_fpr"] == target_fpr)
            paired_diffs.append(
                {
                    "topology": name,
                    "target_fpr": target_fpr,
                    "deficit_diff_degc_min": t_row["integrated_deficit_degc_min"] - p_row["integrated_deficit_degc_min"],
                    "lead_time_diff_min": t_row["mean_lead_time_min"] - p_row["mean_lead_time_min"],
                    "anticipation_rate_diff": t_row["anticipation_rate"] - p_row["anticipation_rate"],
                }
            )

    diffs_arr = np.array([d["deficit_diff_degc_min"] for d in paired_diffs])
    rng_boot = np.random.default_rng(RNG_SEED)
    n = len(diffs_arr)
    boot_means = np.array([diffs_arr[rng_boot.integers(0, n, n)].mean() for _ in range(N_BOOT)])
    pooled_ci = [float(np.percentile(boot_means, 2.5)), float(np.percentile(boot_means, 97.5))]

    lead_diffs_arr = np.array([d["lead_time_diff_min"] for d in paired_diffs])
    boot_lead = np.array([lead_diffs_arr[rng_boot.integers(0, n, n)].mean() for _ in range(N_BOOT)])
    pooled_lead_ci = [float(np.percentile(boot_lead, 2.5)), float(np.percentile(boot_lead, 97.5))]

    return {
        "table": rows,
        "paired_diffs_transition_minus_preheat": paired_diffs,
        "pooled_deficit_diff_mean": float(diffs_arr.mean()),
        "pooled_deficit_diff_bootstrap_ci95": pooled_ci,
        "pooled_deficit_diff_excludes_zero": bool(pooled_ci[0] > 0 or pooled_ci[1] < 0),
        "pooled_lead_time_diff_mean": float(lead_diffs_arr.mean()),
        "pooled_lead_time_diff_bootstrap_ci95": pooled_lead_ci,
        "pooled_lead_time_diff_excludes_zero": bool(pooled_lead_ci[0] > 0 or pooled_lead_ci[1] < 0),
    }


## (4) Full AUC table

Recomputes every topology x lookahead ROC-AUC cell from the stored FPR/TPR arrays via `sklearn.metrics.auc`, checks whether the transition predictor dominates PreHeat in every cell, and runs a paired Wilcoxon signed-rank test plus bootstrap CI on the AUC gap.

In [ ]:
def compute_full_auc_table() -> dict:
    logger.info("[4] Full AUC table across all topologies x lookaheads x predictors")
    topo_results = method_out["metadata"]["per_topology_results"]
    lookaheads = method_out["metadata"]["lookaheads_min"]

    rows = []
    gaps = []
    for tr in topo_results:
        name = tr["topology"]
        for la in lookaheads:
            roc_b = tr["roc_by_lookahead"][str(la)]["baseline"]
            roc_t = tr["roc_by_lookahead"][str(la)]["transition"]

            fpr_b, tpr_b = np.array(roc_b["fpr"]), np.array(roc_b["tpr"])
            order_b = np.argsort(fpr_b)
            auc_b_recomputed = float(sk_auc(fpr_b[order_b], tpr_b[order_b]))

            fpr_t, tpr_t = np.array(roc_t["fpr"]), np.array(roc_t["tpr"])
            order_t = np.argsort(fpr_t)
            auc_t_recomputed = float(sk_auc(fpr_t[order_t], tpr_t[order_t]))

            rows.append(
                {
                    "topology": name,
                    "lookahead_min": la,
                    "auc_preheat_baseline": auc_b_recomputed,
                    "auc_preheat_baseline_stored": roc_b["auc"],
                    "auc_transition_predictor": auc_t_recomputed,
                    "auc_transition_predictor_stored": roc_t["auc"],
                    "auc_gap_transition_minus_preheat": auc_t_recomputed - auc_b_recomputed,
                    "transition_dominates": bool(auc_t_recomputed > auc_b_recomputed),
                }
            )
            gaps.append(auc_t_recomputed - auc_b_recomputed)
            logger.info(f"  [{name}] lookahead={la}min: baseline={auc_b_recomputed:.3f} transition={auc_t_recomputed:.3f} gap={gaps[-1]:+.3f}")

    gaps_arr = np.array(gaps)
    n_cells = len(gaps_arr)
    n_dominant = int(np.sum(gaps_arr > 0))

    wilcoxon_stat, wilcoxon_p = stats.wilcoxon(gaps_arr, alternative="greater")

    rng_boot = np.random.default_rng(RNG_SEED)
    boot_means = np.array([gaps_arr[rng_boot.integers(0, n_cells, n_cells)].mean() for _ in range(N_BOOT)])
    gap_ci = [float(np.percentile(boot_means, 2.5)), float(np.percentile(boot_means, 97.5))]

    return {
        "table": rows,
        "n_cells": n_cells,
        "n_cells_transition_dominates": n_dominant,
        "mean_auc_gap": float(gaps_arr.mean()),
        "paired_bootstrap_ci95_gap": gap_ci,
        "gap_excludes_zero": bool(gap_ci[0] > 0 or gap_ci[1] < 0),
        "wilcoxon_signed_rank_statistic": float(wilcoxon_stat),
        "wilcoxon_p_value_one_sided_greater": float(wilcoxon_p),
        "omnibus_dominance_confirmed_all_cells": bool(n_dominant == n_cells),
    }


## Run all four metric families

Same sequence as `eval.py`'s `main()`.

In [ ]:
logger.info("=== Evaluation: realism, corrected CI/power, graded comfort, full AUC table ===")

realism = compute_realism_table()
ci_power = compute_ci_power_table()
comfort = compute_comfort_proxy_table()
auc_table = compute_full_auc_table()

narrative = (
    f"(1) Realism: synthetic topology closest to real CASAS behavior is "
    f"'{realism['closest_synthetic_to_real_pair']['synthetic']}' (z-distance to "
    f"{realism['closest_synthetic_to_real_pair']['real']} = {realism['closest_synthetic_to_real_pair']['z_distance']:.2f}); "
    f"farthest is '{realism['farthest_synthetic_to_real_pair']['synthetic']}' "
    f"(z-distance to {realism['farthest_synthetic_to_real_pair']['real']} = {realism['farthest_synthetic_to_real_pair']['z_distance']:.2f}). "
    f"(2) The bootstrap CIs reproduce the mini-run's own stored CIs "
    f"({'within Monte Carlo noise' if all(r['ci_reproduces_within_mc_noise'] for r in ci_power['table']) else 'with some drift'}); "
    f"power analysis at n={ci_power['table'][0]['n_topologies']} topologies shows MDE~{ci_power['table'][0]['mde_n3_at_80pct_power_pp']:.1f}pp at "
    f"FPR={ci_power['table'][0]['target_fpr']}, and would need n~{ci_power['table'][0]['n_required_for_5pp_effect_at_80pct_power']:.0f} independent "
    f"topology/household conditions to reliably detect a 5pp mean savings effect at 80% power. "
    f"(3) The graded comfort proxy finds a pooled integrated-temperature-deficit difference (transition-PreHeat) of "
    f"{comfort['pooled_deficit_diff_mean']:.1f} degC*min (bootstrap 95% CI {comfort['pooled_deficit_diff_bootstrap_ci95']}, "
    f"{'excludes zero' if comfort['pooled_deficit_diff_excludes_zero'] else 'includes zero'}) and a pooled realized-lead-time "
    f"difference of {comfort['pooled_lead_time_diff_mean']:.1f} min (CI {comfort['pooled_lead_time_diff_bootstrap_ci95']}, "
    f"{'excludes zero' if comfort['pooled_lead_time_diff_excludes_zero'] else 'includes zero'}). "
    f"(4) The {auc_table['n_cells']}-cell AUC table shows the transition predictor dominates PreHeat's ROC-AUC in "
    f"{auc_table['n_cells_transition_dominates']}/{auc_table['n_cells']} cells (mean gap {auc_table['mean_auc_gap']:+.3f}, "
    f"paired bootstrap 95% CI {auc_table['paired_bootstrap_ci95_gap']}, Wilcoxon one-sided p={auc_table['wilcoxon_p_value_one_sided_greater']:.2e})."
)
logger.info(narrative)
print("\n" + narrative)


## Results

Key tables and a visualization summarizing all four metric families.

In [ ]:
print("=== (1) Realism table: synthetic topologies vs real CASAS houses ===")
for r in realism["table"]:
    print(f"  {r['row']:32s} occ_frac={r['occupied_fraction']:.3f}  mean_dwell={r['mean_dwell_min']:6.1f}min  entropy={r['transition_entropy_bits']:.2f}bits")
print(f"  closest synthetic-real pair:  {realism['closest_synthetic_to_real_pair']}")
print(f"  farthest synthetic-real pair: {realism['farthest_synthetic_to_real_pair']}")

print("\n=== (2) Corrected bootstrap CI + power analysis ===")
for r in ci_power["table"]:
    print(f"  FPR={r['target_fpr']}: mean_savings={r['mean_savings_pct']:6.2f}%  CI95={[round(x,1) for x in r['recomputed_bootstrap_ci95']]}  "
          f"MDE(n={r['n_topologies']})={r['mde_n3_at_80pct_power_pp']:.1f}pp  n_req(5pp)={r['n_required_for_5pp_effect_at_80pct_power']:.1f}")

print("\n=== (3) Graded comfort proxy: pooled paired differences (transition - PreHeat) ===")
print(f"  deficit diff:   mean={comfort['pooled_deficit_diff_mean']:+.1f} degC*min  CI95={[round(x,1) for x in comfort['pooled_deficit_diff_bootstrap_ci95']]}  excludes_zero={comfort['pooled_deficit_diff_excludes_zero']}")
print(f"  lead-time diff: mean={comfort['pooled_lead_time_diff_mean']:+.1f} min       CI95={[round(x,1) for x in comfort['pooled_lead_time_diff_bootstrap_ci95']]}  excludes_zero={comfort['pooled_lead_time_diff_excludes_zero']}")

print("\n=== (4) Full AUC table ===")
for r in auc_table["table"]:
    print(f"  {r['topology']:24s} lookahead={r['lookahead_min']:3d}min  PreHeat={r['auc_preheat_baseline']:.3f}  transition={r['auc_transition_predictor']:.3f}  gap={r['auc_gap_transition_minus_preheat']:+.3f}")
print(f"  dominance: {auc_table['n_cells_transition_dominates']}/{auc_table['n_cells']} cells, mean gap={auc_table['mean_auc_gap']:+.3f}, "
      f"Wilcoxon p={auc_table['wilcoxon_p_value_one_sided_greater']:.2e}")

# -- visualization --
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

# panel 1: realism z-distance heatmap (synthetic topologies x CASAS houses)
dist_rows = realism["synthetic_to_real_distance_matrix"]
synth_names = [r["synthetic_topology"] for r in dist_rows]
house_names = list(dist_rows[0]["distances_to_house"].keys())
dist_grid = np.array([[r["distances_to_house"][h] for h in house_names] for r in dist_rows])
im = axes[0].imshow(dist_grid, cmap="viridis_r", aspect="auto")
axes[0].set_xticks(range(len(house_names)), house_names, rotation=45, ha="right")
axes[0].set_yticks(range(len(synth_names)), synth_names)
axes[0].set_title("(1) Synthetic-to-real z-distance")
for i in range(dist_grid.shape[0]):
    for j in range(dist_grid.shape[1]):
        axes[0].text(j, i, f"{dist_grid[i, j]:.1f}", ha="center", va="center", color="white", fontsize=8)
fig.colorbar(im, ax=axes[0], shrink=0.8)

# panel 2: AUC by topology/lookahead, PreHeat vs transition predictor
labels = [f"{r['topology'][:8]}\n{r['lookahead_min']}min" for r in auc_table["table"]]
x = np.arange(len(labels))
width = 0.35
axes[1].bar(x - width / 2, [r["auc_preheat_baseline"] for r in auc_table["table"]], width, label="PreHeat")
axes[1].bar(x + width / 2, [r["auc_transition_predictor"] for r in auc_table["table"]], width, label="transition")
axes[1].set_xticks(x, labels, fontsize=7)
axes[1].set_ylabel("ROC-AUC")
axes[1].set_title("(4) AUC: PreHeat vs transition predictor")
axes[1].axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
axes[1].legend(fontsize=8)

# panel 3: pooled comfort-proxy differences with bootstrap CIs
metrics = ["deficit diff\n(degC*min)", "lead-time diff\n(min)"]
means = [comfort["pooled_deficit_diff_mean"], comfort["pooled_lead_time_diff_mean"]]
los = [comfort["pooled_deficit_diff_bootstrap_ci95"][0], comfort["pooled_lead_time_diff_bootstrap_ci95"][0]]
his = [comfort["pooled_deficit_diff_bootstrap_ci95"][1], comfort["pooled_lead_time_diff_bootstrap_ci95"][1]]
yerr = np.array([[m - lo, hi - m] for m, lo, hi in zip(means, los, his)]).T
axes[2].bar(metrics, means, yerr=np.abs(yerr), capsize=6, color=["#d62728", "#1f77b4"])
axes[2].axhline(0.0, color="black", linewidth=0.8)
axes[2].set_title("(3) Pooled comfort-proxy diff\n(transition - PreHeat), 95% CI")

plt.tight_layout()
plt.show()
